[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusicalInformatics/miws26/blob/main/expressive_rendering/05_render_performance.ipynb)

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

HOME_DIR = "."

if IN_COLAB:
    import os
    import sys

    # Install system dependency (FluidSynth executable)
    ! apt-get update -qq
    ! apt-get install -y -qq fluidsynth

    # Install python packages
    ! pip install git+https://github.com/CPJKU/partitura.git@develop pyfluidsynth appdirs

    # To be able to access helper modules and data in the repo for this tutorial
    # (not necessary if the jupyter notebook is run locally instead of google colab)
    if not os.path.exists("./miws26"):
        !git clone --recurse-submodules https://github.com/MusicalInformatics/miws26.git
    sys.path.insert(0, "/content/miws26/expressive_rendering/")
    HOME_DIR = "/content/miws26/expressive_rendering/"

# The Basis Mixer!

The previous notebooks explored the individual components of the Basis Function Models. In this notebook we will put everything together to render an expressive performance of a piece given its score.

In [ ]:
import numpy as np
from partitura import save_performance_midi, load_musicxml
from partitura.score import expand_grace_notes
import os

from basismixer.performance_codec import get_performance_codec
from helper import load_model, compute_basis_from_xml
from helper.rendering import post_process_predictions, path_to_trained_models

import warnings

warnings.filterwarnings("ignore", module="partitura")

## 0. Select a piece and trained models
We start by defining the piece we want to render (here is an example MusicXML file, but you can set your own!)

In [ ]:
xml_fn = os.path.join(HOME_DIR, "sample_data", "musicxml", "Mozart_K265_1.musicxml")
# xml_fn = os.path.join(HOME_DIR, "generation_task", "mussorgsky-pictures-at-an-exhibition-06-no-3-tuileries.musicxml")

Now we choose the trained models. You can use either

* **your own model**, trained in the previous notebook (`path_to_trained_models` returns its folder; if you have not trained a model, it returns the folder of our model), or
* **our model**, which we trained on the Vienna 4x22 dataset (in `sample_data/models`). It consists of two models: a recurrent model that predicts the parameters shared by the notes of an onset (`velocity_trend`, `beat_period_standardized`, `beat_period_mean`, `beat_period_std`), and a feed forward model for the parameters of each note (`velocity_dev`, `timing`, `articulation_log`).

Each model is saved in its own folder, with its configuration (`config.json`) and its weights (`best_model.pth`).

In [ ]:
# Choose one of the two:
models_dir = path_to_trained_models()  # your model, trained in the previous notebook
# models_dir = os.path.join(HOME_DIR, "sample_data", "models")  # our model

## 1. Load the Predictive Model

The function `load_model` looks for the configuration files and the saved model parameters in `models_dir`. This function will return an instance of `FullPredictiveModel`, which is a conveniece *meta-model* which contains a list of the individual models (in `model.models`).

In [ ]:
model, predicted_parameter_names = load_model(models_dir)

In [ ]:
# Your values for the expressive parameters. For example, the tempo
# (`beat_period_mean` is the duration of a beat in seconds) and how much it varies
beats_per_minute = 120
my_values = {
    "beat_period_mean": 60.0 / beats_per_minute,
    "beat_period_std": 0.01,
}

# True: your values replace the predictions of the model.
# False: your values are only used for the parameters that the model does not predict.
override_model_preds = True

## 2. Computing the Score Representation.

We load the input MusicXML file specified above and compute the input score representation using the Basis Functions specified by the model (in `model.input_names`). By default, `compute_basis_from_xml` expands the grace notes according to their type (an *acciaccatura* starts before the beat and takes time from the previous note, an *appoggiatura* starts on the beat and takes time from the main note), and removes double notes, i.e., notes with the same pitch and onset written in two voices, which a pianist cannot play twice (and would be a problem for a mechanical piano). You can change this with the arguments `grace_notes` (`"expand"` or `"remove"`) and `double_notes` (`"remove"` or `"keep"`).

In [ ]:
# Load MusicXML file
basis, part = compute_basis_from_xml(xml_fn, model.input_names)

## 3. Make Predict Performance

We make predictions of the performance using the models. The `predict` method of `FullPredictiveModel` combines the predictions of the individual predictive models into a single output structured array, where the field names are the output parameters (specified in `model.output_names`).

The parameters that no model predicts get a default value. With the argument `default_values` you can give your own values (the ones we defined above), and with `override_model_preds=True` they are used *instead of the predictions of the model*. This way you can set, for example, the tempo of the performance, or any other parameter (e.g., `my_values["timing"] = 0` plays all notes of an onset exactly together). Parameters that are not in `default_values` are not affected.

In [ ]:
preds = model.predict(
    basis,
    part.note_array()["onset_beat"],
    default_values=my_values,
    override_model_preds=override_model_preds,
)
post_process_predictions(preds)
print("Field names:", preds.dtype.names)

## 4. Decode predictions

Finally, we decode the predictions using a `PerformanceCodec` and export the performance as a MIDI file. The output of the `decode` method is an instance of `partitura.performance.PerformedPart`.

In [ ]:
perf_codec = get_performance_codec(model.output_names)
predicted_ppart = perf_codec.decode(part, preds)

Finally, we can export the predictions of the model as a MIDI file!

In [ ]:
midi_fn = "my_midi_output.mid"
save_performance_midi(predicted_ppart, midi_fn)

In [ ]:
from helper.rendering import render_midi

render_midi(midi_fn)

In [ ]:
from helper.plotting import plot_predictions

plot_predictions(
    preds,
    part.note_array()["onset_beat"],
    predicted_parameter_names,
)